In [0]:
print("Hello conqueror of worlds")
print(spark.version)

In [0]:
data = [
    ("Vivek", "Canada"),
    ("Test User", "USA")
]

df = spark.createDataFrame(data, ["name", "country"])

display(df)

In [0]:
orders_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/workspace/default/northstar_raw/olist_orders_dataset.csv")

#display(orders_df)

In [0]:
orders_df.printSchema()

In [0]:
print(f"Rows: {orders_df.count()}")
print(f"Columns: {len(orders_df.columns)}")
#print(orders_df.count())

In [0]:
from pyspark.sql import functions as f

orders_df.select([
    f.count(f.when(f.col(c).isNull(), c)).alias(c)
    for c in orders_df.columns
]).display()

In [0]:
def read_olist_csv(filename):
    path = f"/Volumes/workspace/default/northstar_raw/{filename}"

    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .option("mode", "PERMISSIVE")
        .csv(path)
    )

In [0]:
customers_df = read_olist_csv("olist_customers_dataset.csv")

print(f"Rows: {customers_df.count()}")
print(f"Columns: {len(customers_df.columns)}")

customers_df.printSchema()

In [0]:
geolocation_df = read_olist_csv("olist_geolocation_dataset.csv")
order_items_df = read_olist_csv("olist_order_items_dataset.csv")
payments_df = read_olist_csv("olist_order_payments_dataset.csv")
reviews_df =  read_olist_csv("olist_order_reviews_dataset.csv")
products_df = read_olist_csv("olist_products_dataset.csv")
sellers_df =  read_olist_csv("olist_sellers_dataset.csv")
category_translation_df = read_olist_csv("product_category_name_translation.csv")

In [0]:
datasets = {"geolocation":geolocation_df,
            "order_items":order_items_df,
            "payments":payments_df,
            "reviews":reviews_df,
            "products":products_df,
            "sellers":sellers_df,
            "translation":category_translation_df
            }

for name, d in datasets.items():
    print(f"Rows in {name}: {d.count()}")

In [0]:
print(f"products: {products_df.columns}")
print(f"reviews: {reviews_df.columns}")

In [0]:
products_df.printSchema()
reviews_df.printSchema()

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/northstar_raw/"))

In [0]:
reviews_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/northstar_raw/olist_order_reviews_dataset.csv")
)

print(reviews_df.count())

In [0]:
print(dbutils.fs.ls(
    "/Volumes/workspace/default/northstar_raw/olist_order_reviews_dataset.csv"
))

In [0]:
print("Total Rows: ", reviews_df.count())
print("Distinct review IDs: ", reviews_df.select("review_id").distinct().count())
print("Null review IDs: ", reviews_df.filter(f.col("review_id").isNull()).count())

In [0]:
duplicate_reviews = (
    reviews_df.groupby("review_id").count().filter(f.col("count")>1)
)
display(duplicate_reviews.orderBy(f.col("count").desc()))

In [0]:
print("Duplicate review IDs:", duplicate_reviews.count())

print(
    "Rows belonging to duplicate IDs:", duplicate_reviews.select(f.sum("count")).collect()[0][0]
)

In [0]:
reviews_no_multiline = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/northstar_raw/olist_order_reviews_dataset.csv")
)

print("Without multiLine:", reviews_no_multiline.count())
print("With multiLine:", reviews_df.count())

In [0]:
print("Total:", reviews_df.count())

print(
    "Rows with empty review_id:",
    reviews_df.filter(f.trim(f.col("review_id"))=="").count()
)

In [0]:
display(
    reviews_df.filter(f.trim(f.col("review_id"))=="")
)

In [0]:
raw_lines = spark.read.text(
    "/Volumes/workspace/default/northstar_raw/olist_order_reviews_dataset.csv"
)
print("physical lines:", raw_lines.count())

In [0]:
original_reviews = (
    spark.read
    .option("header", True)
    .option("multiline", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv("/Volumes/workspace/default/northstar_raw/olist_order_reviews_dataset.csv")
)
print("Rows:",original_reviews.count())
print("Columns:",len(original_reviews.columns))

Bronze Layer codes

In [0]:
orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_olist_orders")

In [0]:
bronze_orders = spark.table("workspace.default.bronze_olist_orders")

print("Rows:", bronze_orders.count())
print("Columns:", len(bronze_orders.columns))

In [0]:
spark.sql("show tables in workspace.default").show()

In [0]:
spark.sql("select * from workspace.default.bronze_olist_orders limit 5").show()

In [0]:
bronze_datasets = {
    "customers":customers_df,
    "geolocation":geolocation_df,
    "order_items":order_items_df,
    "payments":payments_df,
    "reviews":reviews_df,
    "products":products_df,
    "sellers":sellers_df,
    "translation":category_translation_df,
    "orders": orders_df
}

for name, df in bronze_datasets.items():
    table_name = f"workspace.default.bronze_olist_{name}"

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)

    print(f"Created: {table_name}")

In [0]:
for name in bronze_datasets:
    table_name = f"workspace.default.bronze_olist_{name}"

    row = spark.table(table_name).count()
    columns = len(spark.table(table_name).columns)
    print(f"table :{table_name}, row count: {row}, columns: {columns}")